In [10]:
%%sql
CREATE TABLE IF NOT EXISTS bronze.table_column_details (
    TableName STRING,
    ColumnName STRING,
    ColumnDataType STRING
)
USING DELTA;

StatementMeta(, 75359b6e-1b64-4f05-af5d-b9a6c92fc4aa, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
CREATE TABLE IF NOT EXISTS Bronze.Metadata_Raw_Vaults (
    Bronze_TableName      STRING,
    Bronze_ColumnName     STRING,
    Bronze_DataType       STRING,

    Silver_TableName      STRING,
    Silver_ColumnName     STRING,
    Silver_DataType       STRING,

    IsPrimaryKey          BOOLEAN,
    IsForeignKey          BOOLEAN,
    DV_ObjectType         STRING,

    dv_load_date          TIMESTAMP,
    dv_record_source      STRING,
    dv_batch_id           STRING
)
USING DELTA;

StatementMeta(, 71f3ef03-ea8d-48de-827b-88b214e679a2, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
%%sql
DELETE FROM Bronze.Metadata_Raw_Vaults;

StatementMeta(, 71f3ef03-ea8d-48de-827b-88b214e679a2, 6, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

In [12]:
# ================================
# (1) Libraries
# ================================
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable

# ================================
# (2) Initialize Spark
# ================================
spark = SparkSession.builder.getOrCreate()

# ================================
# (3) Get Tables from Bronze Schema
# ================================
tables_df = spark.sql("SHOW TABLES IN bronze")
tables = [row.tableName for row in tables_df.collect()]

print(f"Total Tables Found: {len(tables)}")

# ================================
# (4) Collect Metadata
# ================================
final_data = []

for table in tables:
    
    full_table = f"bronze.{table}"
    print(f"Processing: {full_table}")
    
    try:
        desc_df = spark.sql(f"DESCRIBE {full_table}")
        
        for row in desc_df.collect():
            col_name = row['col_name']
            data_type = row['data_type']
            
            # Skip metadata rows
            if col_name and not col_name.startswith("#"):
                final_data.append({
                    "TableName": str(table),
                    "ColumnName": str(col_name),
                    "ColumnDataType": str(data_type)
                })
    
    except Exception as e:
        print(f"Error in table {table}: {e}")

# ================================
# (5) Define Schema
# ================================
schema = StructType([
    StructField("TableName", StringType(), True),
    StructField("ColumnName", StringType(), True),
    StructField("ColumnDataType", StringType(), True)
])

# Handle empty case
if not final_data:
    final_data = [{"TableName": "", "ColumnName": "", "ColumnDataType": ""}]

# ================================
# (6) Create DataFrame
# ================================
final_df = spark.createDataFrame(final_data, schema=schema)

# ================================
# (7) Remove Duplicates
# ================================
final_df = final_df.dropDuplicates(["TableName", "ColumnName"])

final_df.show(truncate=False)

# ================================
# (8) Write to Delta Table (MERGE)
# ================================
target_table = "bronze.table_column_details"

# Check if table exists
if spark._jsparkSession.catalog().tableExists(target_table):
    
    print("Table exists → Performing MERGE (Upsert)")
    
    delta_table = DeltaTable.forName(spark, target_table)
    
    delta_table.alias("t").merge(
        final_df.alias("s"),
        "t.TableName = s.TableName AND t.ColumnName = s.ColumnName"
    ).whenMatchedUpdate(
        set={
            "ColumnDataType": "s.ColumnDataType"
        }
    ).whenNotMatchedInsertAll().execute()

else:
    
    print("Table does NOT exist → Creating & inserting data")
    
    final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

print(f"\n✅ Data successfully loaded into: {target_table}")

# ================================
# (9) Validate Result
# ================================
spark.sql(f"SELECT * FROM {target_table}").show(truncate=False)

StatementMeta(, 75359b6e-1b64-4f05-af5d-b9a6c92fc4aa, 14, Finished, Available, Finished, False)

Total Tables Found: 28
Processing: bronze.categories
Processing: bronze.customers
Processing: bronze.employees
Processing: bronze.goods_receipt_notes
Processing: bronze.grn_items
Processing: bronze.gst_invoice_details
Processing: bronze.inventory_cost_layers
Processing: bronze.inventory_stock
Processing: bronze.inventory_transactions
Processing: bronze.pos_sale_items
Processing: bronze.product_tax_mapping
Processing: bronze.products
Processing: bronze.purchase_order_items
Processing: bronze.purchase_orders
Processing: bronze.reorder_rules
Processing: bronze.stock_adjustments
Processing: bronze.stores
Processing: bronze.supplier_invoices
Processing: bronze.supplier_payments
Processing: bronze.supplier_product_rules
Processing: bronze.suppliers
Processing: bronze.table_column_details
Processing: bronze.tax_master
Processing: bronze.units_of_measure
Processing: bronze.warehouse_transfer_items
Processing: bronze.warehouse_transfers
Processing: bronze.warehouses
Processing: bronze.watermark

In [17]:
%%sql

DELETE FROM bronze.table_column_details
WHERE TableName = 'watermark';

StatementMeta(, 75359b6e-1b64-4f05-af5d-b9a6c92fc4aa, 18, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [18]:
%%sql

select * from bronze.table_column_details;

StatementMeta(, 75359b6e-1b64-4f05-af5d-b9a6c92fc4aa, 19, Finished, Available, Finished, False)

<Spark SQL result set with 217 rows and 3 fields>

In [24]:
%%sql
select * from bronze.table_column_details

StatementMeta(, 75359b6e-1b64-4f05-af5d-b9a6c92fc4aa, 25, Finished, Available, Finished, False)

<Spark SQL result set with 217 rows and 3 fields>